In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('../data/rsud_gunungjati.resumeMedis2024.csv')
df.info()

In [ ]:
patterns = [
    r"^diagnosaDokter\[\d+\]\.no$",
    r"^diagnosaDokter\[\d+\]\.jenisDiagnosa.value$",
    r"^diagnosaDokter\[\d+\]\.jenisDiagnosa.label$",
    r"^detailTindakan\[\d+\]\.no$",
    r"^diagnosaDokter\[\d+\]\.norecDiagnosa$",
    r"^detailTindakan\[\d+\]\.norecDiagnosa",
    r"^detailObatResep\[\d+\]\.waktupakai",
    r"^detailObatResep\[\d+\]\.no",
    r"^detailObatResep\[\d+\]\.obat.label$",
    r"^detailObatResep\[\d+\]\.obat.value$",
    r"^detailObatResep\[\d+\]\.jumlah$",
    r"^detailObatResep\[\d+\]\.aturanPakai$",
    r"^detailObatResep\[\d+\]\.waktuPakai$",
    r"^detailObatResep\[\d+\]\.dosis$",
    r"^detailDokterPelayanan",
    r"^dokterDPJP",
    r"^diagnosaDokter\[(\d+)\]\.isLoadBtnDiagnosaDokter$",
    r"^diagnosaDokter\[(\d+)\]\.jenisDiagnosa$",
    r"^detailObatResep\[(\d+)\]\.obat$",
    r"^detailObatResep\[(\d+)\]\.jenisObat$",
    r"^diagnosaDokter\[(\d+)\]\.diagnosaIcd10$",
    r"^detailTindakan\[(\d+)\]\.diagnosaIcd9$",
    r"^detailTindakan\[(\d+)\]\.diagnosaIcd9.value$",
    r"^diagnosaDokter\[(\d+)\]\.diagnosaIcd10.value$",
    r"^detailTindakan\[(\d+)\]\.diagnosaIcd9.value$",
    r"^detailTindakan\[(\d+)\]\.diagnosaIcd9$",
] 

for pattern in patterns:
    df = df[[col for col in df.columns if not re.match(pattern, col)]]

df.info()

In [ ]:
def matching_function(pattern, col):
    match = re.match(pattern, col)
    return match and int(match.group(1))>=3

patterns2 = [
    r"^detailTindakan\[(\d+)\]\.diagnosaIcd9.label$",
    r"^diagnosaDokter\[(\d+)\]\.ketDiagnosaDok$",
    r"^diagnosaDokter\[(\d+)\]\.diagnosaIcd10.label$",
    r"^detailTindakan\[(\d+)\]\.ketTindakanDokter$",
    r"^detailTindakan\[(\d+)\]\.diagnosaIcd9$",
]


for pattern in patterns2:
    if not pattern: break
    df = df[[col for col in df.columns if not matching_function(pattern, col)]]

df.info()


In [ ]:
# remove other unwanted cols
unwanted_cols = ["no","poli","pasien.catatan_terbaru","dokterDPJP","pasien.namapasien", "skalaNyeri", "pasien.tempatlahir", "pasien.suku","pasien.objectjeniskelaminfk", "hasilKonsultasi","diet", "flag", "instruksiPPA", "kdprofile", "statusenabled", "diagnosaDokter[2].0.jenisDiagnosa.value", "diagnosaDokter[2].0.jenisDiagnosa.label", "diagnosaDokter[2].0.keterangan", "diagnosaDokter[2].0.diagnosaa.label", "diagnosaDokter[2].0.diagnosaa.value", "diagnosaDokter[2].0.type", "diagnosaDokter[2].1.jenisDiagnosa.label", "diagnosaDokter[2].1.jenisDiagnosa.value", "diagnosaDokter[2].1.keterangan", "diagnosaDokter[2].1.type", "keteranganVerifikasiDPJP", "tenagaMedis", "detailDokterPelayanan[0].no", "dokterDPJP.label","dokterDPJP.value","pelaksana"]
unwanted_cols2 = ["pasien.noidentitas","pasien.nocm", "detailTindakan[0].isLoadBtnDiagnosaDokter9", "pasien.nocmfk","pasien.nobpjs", "pasien.noasuransilain","pasien.alamatlengkap","pasien.kodepos","pasien.notelepon","pasien.nohp","pasien.namaayah","pasien.namaibu","pasien.email","pasien.agama","pasien.pendidikan",	"pasien.pekerjaan","pasien.isfoto","pasien.filename","pasien.catatan","pasien.isFilterProdukLab","pasien.enabledEMRSimrsLama","statusenabled","noemr","emrpasienfk","id","created_at","updated_at",	"IMT","kondisiKeluar","statusSelesai","totalSkor","alergiReaksiObat","gcs","hasilPenunjang","kesadaran","kondisiKeluarLainya","poli.value","tglperjanjian","pemeriksaanfisiklainnya","statusWA","tidakAdaTurunBeratBadan","turunBeratBadan","asupanMakan","riwayatPenyakitSekarang","alergireaksiobat","hasilkonsultasi","hasilpenunjang","skalaNyeri->label"]

wanted_cols  = ["registrasi.kelompokpasien", "registrasi.asalrujukan","registrasi.namakelas", "pasien.nocm", "pasien.nocmfk", "pasien.tgllahir", "pasien.jeniskelamin", "pasien.umur"]


df = df[[col for col in df.columns if col not in unwanted_cols]]
df = df[[col for col in df.columns if col not in unwanted_cols2]]
df = df[[col for col in df.columns if not (col.startswith("registrasi") and col not in wanted_cols)]]
df = df[[col for col in df.columns if not (col.startswith("pasien.") and col not in wanted_cols)]]
df = df[[col for col in df.columns if not (col.startswith("user_input.") or col.startswith("profile.") or col.startswith("prognosis"))]]

df.info()

# df.groupby('diagnosaDokter[0].diagnosaIcd10.label').size()

In [ ]:
df2 = df.copy()
df2 = df2.dropna(subset=['diagnosaDokter[0].ketDiagnosaDok'])
df2 = df2[~df2['diagnosaDokter[0].ketDiagnosaDok'].str.contains('riw', case=False, na=False)]
df2 = df2[~df2['diagnosaDokter[0].ketDiagnosaDok'].str.contains('post', case=False, na=False)]
df2 = df2[~df2['diagnosaDokter[0].ketDiagnosaDok'].str.contains('pasca', case=False, na=False)]

In [ ]:
def replace_func(toreplace, replacedwith):
    df2.loc[df2['diagnosaDokter[0].ketDiagnosaDok'].str.contains(toreplace, case=False), 'grouped_diagnosa'] = replacedwith

map1 = {
    'atrial fib':'atrial fibrilasi',
    'b20':'human immunodeficiency virus',
    'bp':'bronkopneumonia',
    'bph':'hiperplasia',
    'bactereial':"infeksi bakteri",
    'bacteri':"infeksi bakteri",
    'cap':'pneumonia',
    'cf':'cystic fibrosis',
    'ckr':'cedera kepala',
    'cml':'leukemia',
    'leukem':'leukemia',
    'cedera kepala':'cedera kepala',
    'cerbral in':'cerebral infarction',
    'cerebral i':'cerebral infarction',
    'chole':'cholelithiasis',
    'dhf':'dengue fever',
    'dss':'dengue fever',
    'demam t':'demam tifoid',
    'demensia':'dementia',
    'demam deng':'dengue fever',
    'dengue fever':'dengue fever',
    'edh':'epidural hematoma',
    'fistel':'fistel',
    'gea':'gastroenteritis',
    'Gastric Outlet Obstruction':'gastric outlet obstruction',
    'hhd':'hipertensi',
    'hil':'hernia',
    'him':'hernia',
    'hiv':'human immunodeficiency virus',
    'hnp':'hernia',
    'hematemesis':'hematemesis',
    'hematokezia':'hematoskezia',
    'ich':'intracerebral hemorrhage',
    'ilo':'infeksi luka operasi',
    'ispb':'infeksi saluran pernapasan',
    'ispa':'infeksi saluran pernapasan',
    'isk':'infeksi saluran kemih',
    'infeksi viral':'infeksi virus',
    'kad':'ketoasidosis diabetik',
    'kds':'kejang demam',
    'kdk':'kejang demam',
    'karsinoma':'carcinoma',
    'kolelitiasis':'kolelitiasis',
    'kista':'kista',
    'labio':'labiognatopalatoschizis',
    'limfadenopati':'limfadenopati',
    'mds':'sindrom mielodisplastik',
    'mhi':'mental health inventory',
    'mpn':'myeloproliferative neoplasms',
    'meningo en':'meningoencephalitis',
    'meningoen':'meningoencephalitis',
    'NCB':'neonatus cukup bulan',
    'NKB':'neonatus kurang bulan',
    'vomit':'vomit',
    'febris':'febris',
    'kejang':'kejang',
    'Dispnea':'dyspnea',
    'pericoronitis':'pericoronitis',
    'peritonitis':'peritonitis',
    'rds':'respiratory distress syndrome',
    'rdn':'renal denervation',
    'RRD':'Ablasio retina regmatogenosa',
    'radiculopathy':'radikulopati',
    'radiculopati':'radikulopati',
    'respiratory failure':'respiratory failure',
    'retardasi mental':'retardasi mental',
    'retensi':'retensi',
    'rhinosinus':'rhinosinusitis',
    'sah':"subarachnoid hemorrhage",
    'scc':'carcinoma',
    'snh':'stroke hemoragik',
    'SOL':'space occupying lesion',
    'so filled':'silikon oil filled eye',
    'stt':'soft tissue tumor',
    'schizo':'skizophrenia',
    'sepsis':'sepsis',
    'spond':'spondilosis',
    'TB':'tuberkulosis',
    'TBC':'tuberkulosis',
    'tetanus':'tetanus',
    'toxoplasm':'toxoplasmosis',
    'trombocytopenia':'trombositopenia',
    'trombocitopenia':'trombositopenia',
    'thrombocytopenia':'trombositopenia',
    'thrombositopenia':'trombositopenia',
    'UDT':'undescended testis',
    'Varikokel':'varikokel',
    'vertigo':'vertigo',
    'viral':'infeksi virus',
    'volvulus':'volvulus',
    'volnuis':'volnuis',
    'vulnus':'vulnus',
    'abdomeinal pain':'abdominal pain',
    'abdominal pain':'abdominal pain',
    'abses':'abses',
    'adeno ca':'adenocarcinoma',
    'adenoca':'adenocarcinoma',
    'adenocarcinoma':'adenocarcinoma',
    'adhesi':'adhesi',
    'anemia':'anemia',
    'aortic stenosis':'aortic stenosis',
    'aorta regur':'aorta regurgitasi',
    'appendi':'appendisitis',
    'asfik':'asfiksia',
    'asma':'asma',
    'asthma':'asma',
    'atresi':'atresia',
    'baccterial in':'infeksi bakteri',
    'bacterial n':'infeksi bakteri',
    'bactrial i':'infeksi bakteri',
    'bacterial i':'infeksi bakteri',
    'bakterail i':'infeksi bakteri',
    'bakterial i':'infeksi bakteri',
    'batu ginjal':'batu ginjal',
    'batu staghorn':'batu staghorn',
    'batu ureter':'batu ureter',
    'bone expos':'bone exposed',
    'bradikardi':'bradikardia',
    'bronchopneumonia':'bronkopneumonia',
    'broncopneumonia':'bronkopneumonia',
    'bronkitis':'bronkitis',
    'bronkopenumonia':'bronkopneumonia',
    'bronkopnemonia':'bronkopneumonia',
    'bronkopneumonia':'bronkopneumonia',
    'cad':'coronary artery disease',
    'cedera kepala':'cedera kepala',
    'cerebral infar':'cerebral infark',
    'cerebral infrac':'cerebral infark',
    'chestpain':'chest pain',
    'chest pain':'chest pain',
    'cholecystitis':'cholecystitis',
    'cholelithiasis':'cholelithiasis',
    'cholelitiasis':'cholelithiasis',
    'ckd': 'chronic kidney disease',
    'colic': 'colic',
    'colitis':'colitis',
    'combusio':'luka bakar',
    'combusti':'luka bakar',
    'bile duct':'bile duct',
    'condiloma':'condyloma',
    'condyloma':'condyloma',
    'congenital tisuue':'congenital tissue',
    'congenital tissue':'congenital tissue',
    'corpus alienum':'corpus alienum',
    'crush':'crush injury',
    'dehidrasi':'dehidrasi',
    'dermatitis':'dermatitis',
    'depresi':'depresi',
    'diabetic foot':'diabetic foot',
    'diare':'diare',
    'daire':'diare',
    'dm':'diabetes mellitus',
    'edema pulmo':'edema paru',
    'edema paru':'edema paru',
    'efusi pleura':'efusi pleura',
    'encephalopaty':'ensefalopati',
    'encepalopati':'ensefalopati',
    'encephalopathy':'ensefalopati',
    'ensefalopati':'ensefalopati',
    'epididimoor':'epididimoorchitis',
    'epileps':'epilepsi',
    'fascitisis': 'fascitis',
    'fasitis':'fascitis',
    'fascitis':'fascitis',
    'fasciitis':'fascitis',
    'fibrosis':'fibrosis',
    'finger tip injury':'finger tip injury',
    'fistula':'fistula',
    'fournier gang':'fournier gangrene',
    'fornier gang':'fournier gangrene',
    'gagal nafas':'gagal napas',
    'gagal napas':'gagal napas',
    'bipolar':'bipolar',
    'gastrititis':'gastritis',
    'gastritis':'gastritis',
    'glaucoma':'glaukoma',
    'glaukoma':'glaukoma',
    'gout':'gout arthritis',
    'hemangioma':'hemangioma',
    'hematemesis':'hematemesis',
    'hematosezia':'hematoskezia',
    'hematoscezia':'hematoskezia',
    'hematoskezia':'hematoskezia',
    'hematuria':'hematuria',
    'hemoroid':'hemoroid',
    'hemorrhoid':'hemoroid',
    'haemorrhoid':'hemoroid',
    'hepatitis':'hepatitis',
    'hernia':'hernia',
    'herpes zoster':'herpes zoster',
    'hidrocele':'hidrokel',
    'hidrocephalus':'hidrosefalus',
    'hidrokel':'hidrokel',
    'hidrosefalus':'hidrosefalus',
    'hiperglikemia':'hiperglikemia',
    'hiperkealemia':'hiperkalemia',
    'hiperkalemia':'hiperkalemia',
    'hipercalemia':'hiperkalemia',
    'hiperkoagu':'hiperkoagulasi',
    'hipertensi':'hipertensi',
    'hipertrofi':'hipertrofi',
    'hipoalbumin':'hipoalbumin',
    'hipoglikemi':'hypoglycemia',
    'hiponatremi':'hiponatremi',
    'hipospadia':'hipospadia',
    "hircsprung" :"hirschsprung disease",
    "hirschsprung":"hirschsprung disease",
    "hirschsprung"       :"hirschsprung disease",
    "hirscprung":"hirschsprung disease",
    "hirsprung":"hirschsprung disease",
    "hisrprung":"hirschsprung disease",
    'hydrocele':'hydrocele',
    'hydronefrosis':'hydronefrosis',
    'hidronefrosis':'hidronefrosis',
    'hypertropic':'hipertrofi',
    'hypertrophy':'hipertrofi',
    'hypertrophic':'hipertrofi',
    'hypertofi':'hipertrofi',
    'hypertrofi':'hipertrofi',
    "hypoglicaemia" :"hypoglycemia",  
    "hypoglikemia":"hypoglycemia",      
    "hypoglycemia":"hypoglycemia",
    'hypocalcemia':'hypokalemia',
    'hypokalaemia':'hypokalemia',
    'hypokalemia':'hypokalemia',
    'hypospadia' :'hypospadia',
    'icteric neonat': 'ikterik neonatorum',
    'ikterik neonat': 'ikterik neonatorum',
    'ileus loc':'ileus local',
    'ileus lok':'ileus lokal',
    'ileus foc':'ileus focal',
    'ileus obs':'ileus obstruksi',
    'impaksi':'impaksi',
    'infeksi saluran': 'infeksi saluran nafas',
    'infeksi bakteri':'infeksi bakteri',
    'infeksi viral':'infeksi virus',
    'infeksi virus':'infeksi virus',
    'insufisensi renal':'insufisiensi renal',
    'insufisiensi renal':'insufisiensi renal',
    'karsinoma':'karsinoma',
    'kista':'kista',
    'kolik':'kolik',
    'kolitis':'kolitis',
    'limfadenitis':'limfadenitis',
    'limfadenopati':'limfadenopati',
    'limfoma':'lymphoma',
    'luka bakar':'luka bakar',
    'lymphoma':'lymphoma',
    'malunion':'fraktur',
    'melanoma':'melanoma',
    'melena':'melena',
    'meningitis':'meningitis',
    'necroti':'necrotic',
    'nefrolitihiasis':'nefrolithiasis',
    'nefrolitiasis':'nefrolithiasis',
    'nefrolithiasis': 'nefrolithiasis',
    'dislokasi':'dislokasi',
    'neonat':'neonatal',
    'fraktur':'fraktur',
    'fractu': 'fraktur',
    'dyspne': 'dyspnea',
    'dysn': 'dyspnea',
    'dyspenu': 'dyspnea',
    'dyspnea':"dyspnea",
    'open degloving':'degloving',
    'open dislokasi':'dislokasi',
    'open faktur': 'fraktur',
    'open fr': 'fraktur',
    'open wound': 'open wound',
    'orchitis':'orchitis',
    'pancreatitis':'pancreatitis',
    'pankreatitis':'pancreatitis',
    'parkinson':'parkinson disease',
    'penkes':'penurunan kesadaran',
    'penurunan kesadaran':'penurunan kesadaran',
    'periodic parali':'periodik paralitis',
    'periodik parali':'periodik paralitis',
    'perotinitis':'peritonitis',
    'peritonitis':'peritonitis',
    "pharingitis":'pharingitis',
    'pnemonia':'pneumonia',
    'pneumina':'pneumonia',
    'penumonia':'pneumonia',
    'pneumonua':'pneumonia',
    'pneumotho':'pneumothorax',
    'pneumotor':'pneumothorax',
    'pneumonia':'pneumonia',
    'polip':'polip',
    'ppok':'penyakit paru obstruktif kronis',
    'prolaps':'prolaps',
    'radiculopa':'radikulopati',
    'radikulopa':'radikulopati',
    'redetachmen':'redetachment OD',
    'resp fail':'respiratory failure',
    'respiratoruy failur':'respiratory failure',
    'retensi urin':'retensi urine',
    'selulitis':'sellulitis',
    'septic':'sepsis',
    'sepsis':'sepsis',
    'sequalae stroke': 'sequele stroke',
    'sequele stroke':'sequele stroke',
    'severe sep':'severe septic',
    'dyspep':'dyspepsia',
    'Dyspesia':'dyspepsia',
    'dispep':'dyspepsia',
    'sinusi':'sinusitis',
    'sirosis hepa':'sirosis hepatis',
    'skinloss':'skin loss',
    'skin loss': 'skin loss',
    'skizo':'skizophrenia',
    'schizo':'skizophrenia',
    'snak bite':'snake bite',
    'snake bite':'snake bite',
    'striktur CBD':'striktur CBD',
    'strikture uret': 'striktur urethra',
    'striktur uret':'striktur urethra',
    'stroke haemoragik':'stroke hemoragik',
    'stroke hemoragik': 'stroke hemoragik',
    'stroke infak':'stroke infark',
    'stroke infarc':'stroke infark',
    'stroke infark':'stroke infark',
    'syok hipovolemik':'syok hipovolemik',
    'tb paru':'tuberkulosis',
    'tetanus':'tetanus',
    'foid fever':'demam tifoid',
    'tong tie':'tongue tie',
    'tongue tie': 'tongue tie',
    'traumatic amputa':'traumatic amputation',
    'trombocitopeni':'trombositopenia',
    'trombositopenia':'trombositopenia',
    'tumor':'tumor',
    "ulcus diabeti": "ulkus diabetikum",
    "ulcus dm": "ulkus diabetikum",
    "ulcus gan":"ulkus gangrene",
    "ulcus cronis":"ulkus cronis",
    "ulcus corena":"ulkus kornea",
    "ulkus dm": "ulkus diabetikum",
    "ulkus diabeti": "ulkus diabetikum",
    "ulkus gan":"ulkus gangrene",
    "ulkus kornea": "ulkus kornea",
    "ulkus necrotik": "ulkus nekrotik",
    "ulkus nekrotik":"ulkus nekrotik",
    "ulkus pedis":"ulkus pedis",
    "ulkus pepticum": "ulkus pepticum",
    "unidentified snake bite": "unidentified snake bite",
    "union fr": "fraktur",
    "angina pectoris": "angina pectoris",
    "angina":"angina pectoris",
    "Varikokel":"varicocle",
    "varicoc": "varicocle",
    "Varices": "varises",
    "varises": "varises",
    "Vertigo": "vertigo",
    "Viral inf": "viral Infection",
    "vomit":"vomit",
    "vulnus":"vulnus",
    "appertum": "vulnus",
    'STEMI': 'st-elevation myocardial infarction',
}

for k,v in map1.items():
    replace_func(k,v)

In [ ]:
# Hitung jumlah kemunculan masing-masing nilai
group_counts = df2['grouped_diagnosa'].value_counts()

# Filter: ambil hanya nilai yang muncul >= 5 kali
df2 = df2[df2['grouped_diagnosa'].isin(group_counts[group_counts >= 5].index)]

df2.groupby('grouped_diagnosa').size()


In [ ]:
df2 = df2.rename(columns={'diagnosaDokter[0].diagnosaIcd10.label' : 'icd10_label'})
df2 = df2.rename(columns={'detailTindakan[0].diagnosaIcd9.label' : 'icd9_label'})
df2 = df2.rename(columns={'diagnosaDokter[0].ketDiagnosaDok' : 'diagnosaDokter[0].keterangan'})
df2.info()

In [ ]:
df2_export = df2[['grouped_diagnosa', 'diagnosaDokter[0].keterangan', 'icd10_label', 'pasien.jeniskelamin','pasien.umur','registrasi.kelompokpasien','registrasi.namakelas', 
                  'icd9_label', 'detailTindakan[0].ketTindakanDokter', 'tinggiBadan', 'beratBadan', 'tekananDarah', 'nadi', 'suhu','pernapasan', 'SPO2', 'terapi', 'riwayatPenyakit', 'pemeriksaanFisikLain',
                  'pemeriksaPenunjang', 'hasilIntruksi']]
df2_export.to_csv('resume_medis_extract.csv')